# TFLite Conversion for the ZC702 Board

Converts the trained Keras model to a quantised `.tflite` file for on-device inference on the ZC702. Two real problems had to be solved to get a correct conversion (both explained inline below, not just fixed silently):

1. **Version mismatch** - the model was saved with `keras_version 2.10.0` exactly; this notebook must run in a matching TF 2.10 environment (`C:\tf210env`), not the main TF 2.18 environment, or the load/convert step produces a file the board's runtime can't run correctly.
2. **Augmentation layer leaking into the graph** - the model's `safe_augmentation` layer (random rotation/zoom/contrast) is training-only, but its ops (`RngReadAndSkip`, `ImageProjectiveTransformV3`, ...) were still being traced into the exported graph even with `training=False`. Root cause and fix are in the code cell below.

Run this notebook's kernel as `tf210env` (Kernel -> Change Kernel).

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path

print("TensorFlow:", tf.__version__)

PROJECT_ROOT = Path("../..").resolve()
EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / "06_cross_view_mobilenetv2"
DEPLOY_DIR = PROJECT_ROOT / "fpga" / "deployment"
DEPLOY_DIR.mkdir(parents=True, exist_ok=True)

# Change this to convert a different checkpoint (e.g. the frozen-backbone model instead)
MODEL_PATH = EXPERIMENT_DIR / "best_mobilenetv2_crossview_finetuned.keras"
OUTPUT_NAME = "mobilenetv2_crossview_finetuned_int8.tflite"
IMG_SIZE = (224, 224)

TensorFlow: 2.10.0


## Load model and fix the augmentation-layer leak

`Sequential.layers` is a read-only computed property, so `aug_layer.layers.clear()` silently does nothing (confirmed by printing before/after during the original debugging session - identical). The fix that actually works: monkey-patch the layer's bound `.call` method directly to identity, before tracing/conversion.

In [2]:
model = tf.keras.models.load_model(MODEL_PATH)

aug_layer = None
for layer in model.layers:
    if "augmentation" in layer.name.lower():
        aug_layer = layer
        break
if aug_layer is None:
    raise ValueError("Could not find augmentation layer.")

aug_layer.call = lambda inputs, training=None: inputs
print("Monkey-patched augmentation layer:", aug_layer.name)

Monkey-patched augmentation layer: safe_augmentation


## Representative dataset and INT8 conversion

120 real training images (not test images, to avoid any calibration/evaluation leakage) drive the post-training INT8 quantisation. Input/output stay `float32` (the common "integer-only compute, float I/O" scheme) - the model's own `mobilenet_v2.preprocess_input` expects raw 0-255 pixels, not pre-normalised input.

In [3]:
unified_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "unified_multiview_metadata.csv")
rep_paths = unified_df[unified_df["split"] == "train"].sample(n=120, random_state=42)["image_path"].tolist()


def representative_dataset():
    for path in rep_paths:
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, IMG_SIZE)
        img = tf.cast(img, tf.float32)
        yield [tf.expand_dims(img, 0)]


converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
tflite_model = converter.convert()

output_path = DEPLOY_DIR / OUTPUT_NAME
output_path.write_bytes(tflite_model)
print(f"Saved {output_path} ({len(tflite_model)/1e6:.2f} MB)")

C:\tf210env\lib\site-packages\tensorflow\lite\python\convert.py:766: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn("Statistics for quantized inputs were expected, but not "


Saved D:\MSC_PROJECT\fpga\deployment\mobilenetv2_crossview_finetuned_int8.tflite (2.72 MB)


## Verification: no leaked ops, and Keras vs. TFLite agreement

Two checks, both real, neither assumed: (1) the binary shouldn't contain any of the augmentation op names that leaked in before the fix; (2) TFLite's predictions should closely match the original Keras model's on held-out images.

In [4]:
bad_ops = [b"RngReadAndSkip", b"ImageProjectiveTransformV3", b"AdjustContrastv2", b"StatelessRandomUniformV2"]
found_bad = [op for op in bad_ops if op in tflite_model]
if found_bad:
    print("WARNING: found leaked augmentation ops:", found_bad)
else:
    print("No leaked augmentation ops found - clean conversion.")

interp = tf.lite.Interpreter(model_path=str(output_path))
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]
print("Input:", inp["shape"], inp["dtype"])
print("Output:", out["shape"], out["dtype"])

test_df = unified_df[unified_df["split"] == "test"].sample(n=50, random_state=7)
agree = 0
for _, row in test_df.iterrows():
    img = tf.io.read_file(row["image_path"])
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.cast(tf.image.resize(img, IMG_SIZE), tf.float32)
    x = tf.expand_dims(img, 0)

    keras_pred = int(np.argmax(model(x, training=False).numpy()[0]))

    interp.set_tensor(inp["index"], x.numpy())
    interp.invoke()
    tflite_pred = int(np.argmax(interp.get_tensor(out["index"])[0]))

    agree += int(keras_pred == tflite_pred)

print(f"Keras vs TFLite-INT8 agreement on 50 held-out images: {agree}/50 = {100*agree/50:.1f}%")

No leaked augmentation ops found - clean conversion.
Input: [  1 224 224   3] <class 'numpy.float32'>
Output: [ 1 10] <class 'numpy.float32'>


Keras vs TFLite-INT8 agreement on 50 held-out images: 50/50 = 100.0%


## Why the board needs a custom-built `tflite_runtime`, not the official PyPI wheel

The ZC702's Cortex-A9 has `vfpv3` but no `vfpv4`/FMA (`cat /proc/cpuinfo` on the board confirms this). The official `tflite_runtime` wheel is built assuming `vfpv4`, which causes an `Illegal instruction` crash at `interpreter.invoke()` on this exact chip family - a known issue shared with PYNQ-Z1/Z2 boards (same SoC). The fix (building `tflite_runtime` from source, natively on the board) is a board-side operation, not part of this conversion notebook - see `fpga/notebooks/04_zc702_board_check.ipynb` and `fpga/README.md` for the full story.